### Imports

In [1]:
import json
import numpy as np
import pandas as pd
import pingouin as pg
import seaborn as sn

print(pg.__version__) # 0.5.3
print(pd.__version__) # 2.0.3
print(np.__version__) # 1.24.3
print(sn.__version__) # 0.13.0

from utils_MS import *

# %load_ext autotime

0.5.3
2.2.2
1.26.4
0.13.2


/home/ealvarez/miniconda3/envs/graph_matching/lib/python3.10/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.3, the latest is 0.6.1.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


In [2]:
# def run(exp):

### Parameters

In [3]:
file = open("exp.json")
experiment = json.load(file)
exp = experiment["exp"]

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

apply_transformation = params["apply_transformation"]
print("Apply transformation:", apply_transformation)

threshold_corr = params["threshold_corr"]
print("Threshold corr:\t", threshold_corr)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups_id_no = params["groups_id_no"]
print("Groups id (no):\t", groups_id_no)

Exp:		 exp13
Data variations: ['none']
Apply transformation: False
Threshold corr:	 0.5
Groups id:	 ['AR', 'BC', 'BPH', 'CKD14', 'CKD5', 'CRS', 'ED', 'HC', 'LP', 'LPRD', 'LSNB', 'OSA', 'PCa', 'PD', 'PSG', 'RCC', 'SGB', 'health']
Subgroups id:	 {'AR': ['1', '2'], 'BC': ['1', '2'], 'BPH': ['1', '2'], 'CKD14': ['1', '2'], 'CKD5': ['1', '2'], 'CRS': ['1', '2'], 'ED': ['1', '2'], 'HC': ['1', '2'], 'LP': ['1', '2'], 'LPRD': ['1', '2'], 'LSNB': ['1', '2'], 'OSA': ['1', '2'], 'PCa': ['1', '2'], 'PD': ['1', '2'], 'PSG': ['1', '2'], 'RCC': ['1', '2'], 'SGB': ['1', '2'], 'health': ['1', '2']}
Groups id (no):	 ['Blank', 'QC', 'Std']


In [4]:
# Remove
# groups_id = ["OSA"]

### Load dataset

In [5]:
# read raw data
df_join_raw = pd.read_csv("experiments/input/{}_raw.csv".format(exp), index_col=0)
df_join_raw

,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_2.1,AR_2.2,AR_2.3,BC_1.1,...,SGB_1.3,SGB_2.1,SGB_2.2,SGB_2.3,health_1.1,health_1.2,health_1.3,health_2.1,health_2.2,health_2.3
0,1.0,69.99951,Unknown,0.067434,0.524123,-0.866715,0.407143,-0.406209,0.009354,0.310919,...,-1.070179,-0.944405,-1.070179,0.243862,0.213709,-0.178399,-0.301910,-0.167625,-0.189232,-0.178399
1,1.0,70.04025,Unknown,2.617854,3.259370,4.805380,3.076543,2.158068,2.745557,2.617659,...,2.074975,2.584545,2.074975,2.450491,2.697953,2.316318,2.421034,2.327056,2.305521,2.316318
2,1.0,70.04151,Unknown,1.088222,1.732480,1.674824,1.235654,0.836471,1.315006,1.484309,...,1.811865,1.974349,1.811865,0.854473,1.301124,1.092894,1.155690,1.098620,1.087138,1.092894
3,1.0,70.04908,Unknown,1.880941,2.504997,2.535388,1.880900,1.544437,2.014078,2.423544,...,0.706776,1.856629,0.706776,1.714065,2.358324,2.080343,1.900361,2.088160,2.072484,2.080343
4,1.0,70.06267,Unknown,1.659528,2.097781,2.593391,1.309826,1.316238,1.668676,2.019882,...,1.153538,1.286182,1.153538,1.050297,2.077151,1.793208,1.525407,1.801015,1.785358,1.793208
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,1.0,732.79951,Unknown,0.892378,3.759797,0.585675,1.039707,0.346959,0.665817,3.175910,...,0.195989,2.321735,0.195989,0.663823,1.091110,0.643823,0.487232,0.656200,0.631372,0.643823
5440,1.0,748.76437,Unknown,0.916754,1.620635,0.718502,1.221477,0.400736,1.054753,3.072149,...,0.297500,0.758849,0.297500,0.759580,1.002615,0.575805,0.695502,0.587540,0.564006,0.575805
5441,1.0,794.79590,Unknown,1.151250,2.041729,2.984408,1.105385,0.734599,1.412716,3.116183,...,3.053481,0.755768,3.053481,0.888324,1.291584,0.955433,1.061403,0.964905,0.945910,0.955433
5442,1.0,800.81295,Unknown,1.478938,2.421294,3.121924,1.400182,1.092336,1.761611,3.365476,...,3.010984,2.673166,3.010984,1.239034,1.662417,1.343335,1.420552,1.352313,1.334307,1.343335


In [6]:
# get metadata
df_join_raw_metadata = df_join_raw.iloc[:, :2]
df_join_raw_metadata

,Average Rt,Average Mz
0,1.0,69.99951
1,1.0,70.04025
2,1.0,70.04151
3,1.0,70.04908
4,1.0,70.06267
...,...,...
5439,1.0,732.79951
5440,1.0,748.76437
5441,1.0,794.79590
5442,1.0,800.81295


In [7]:
# filter by samples
columns_sample = [column for column in df_join_raw.columns if column.split("_")[0] not in groups_id_no]
df_join_raw_intensity = df_join_raw.loc[:, columns_sample]
df_join_raw_intensity = df_join_raw_intensity.iloc[:, 3:]
df_join_raw_intensity

,AR_1.1,AR_1.2,AR_1.3,AR_2.1,AR_2.2,AR_2.3,BC_1.1,BC_1.2,BC_1.3,BC_2.1,...,SGB_1.3,SGB_2.1,SGB_2.2,SGB_2.3,health_1.1,health_1.2,health_1.3,health_2.1,health_2.2,health_2.3
0,0.067434,0.524123,-0.866715,0.407143,-0.406209,0.009354,0.310919,0.384794,0.934567,-0.608435,...,-1.070179,-0.944405,-1.070179,0.243862,0.213709,-0.178399,-0.301910,-0.167625,-0.189232,-0.178399
1,2.617854,3.259370,4.805380,3.076543,2.158068,2.745557,2.617659,2.637159,1.976523,2.768182,...,2.074975,2.584545,2.074975,2.450491,2.697953,2.316318,2.421034,2.327056,2.305521,2.316318
2,1.088222,1.732480,1.674824,1.235654,0.836471,1.315006,1.484309,1.545443,2.006744,0.707240,...,1.811865,1.974349,1.811865,0.854473,1.301124,1.092894,1.155690,1.098620,1.087138,1.092894
3,1.880941,2.504997,2.535388,1.880900,1.544437,2.014078,2.423544,2.542821,2.784935,0.742334,...,0.706776,1.856629,0.706776,1.714065,2.358324,2.080343,1.900361,2.088160,2.072484,2.080343
4,1.659528,2.097781,2.593391,1.309826,1.316238,1.668676,2.019882,2.131893,2.546386,0.675713,...,1.153538,1.286182,1.153538,1.050297,2.077151,1.793208,1.525407,1.801015,1.785358,1.793208
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,0.892378,3.759797,0.585675,1.039707,0.346959,0.665817,3.175910,2.323982,0.982068,3.403323,...,0.195989,2.321735,0.195989,0.663823,1.091110,0.643823,0.487232,0.656200,0.631372,0.643823
5440,0.916754,1.620635,0.718502,1.221477,0.400736,1.054753,3.072149,2.666972,0.952674,0.381335,...,0.297500,0.758849,0.297500,0.759580,1.002615,0.575805,0.695502,0.587540,0.564006,0.575805
5441,1.151250,2.041729,2.984408,1.105385,0.734599,1.412716,3.116183,0.845104,1.012288,3.827945,...,3.053481,0.755768,3.053481,0.888324,1.291584,0.955433,1.061403,0.964905,0.945910,0.955433
5442,1.478938,2.421294,3.121924,1.400182,1.092336,1.761611,3.365476,2.851395,1.291999,3.981077,...,3.010984,2.673166,3.010984,1.239034,1.662417,1.343335,1.420552,1.352313,1.334307,1.343335


In [8]:
df_join_raw_intensity.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5444 entries, 0 to 5443
Columns: 108 entries, AR_1.1 to health_2.3
dtypes: float64(108)
memory usage: 4.5 MB


In [9]:
check_dataset(df_join_raw_intensity)

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 24312
Count zero:	 1
Count positive:	 563639
Count greater than 1:	 478820
Count less than -1:	 3371


### Generate graphs

In [10]:
# Transformation (log10)

if apply_transformation:
	df_join_raw_log = log10_global(df_join_raw_intensity)
else:
	df_join_raw_log = df_join_raw_intensity.copy()
df_join_raw_log.head()

,AR_1.1,AR_1.2,AR_1.3,AR_2.1,AR_2.2,AR_2.3,BC_1.1,BC_1.2,BC_1.3,BC_2.1,...,SGB_1.3,SGB_2.1,SGB_2.2,SGB_2.3,health_1.1,health_1.2,health_1.3,health_2.1,health_2.2,health_2.3
0,0.067434,0.524123,-0.866715,0.407143,-0.406209,0.009354,0.310919,0.384794,0.934567,-0.608435,...,-1.070179,-0.944405,-1.070179,0.243862,0.213709,-0.178399,-0.301910,-0.167625,-0.189232,-0.178399
1,2.617854,3.259370,4.805380,3.076543,2.158068,2.745557,2.617659,2.637159,1.976523,2.768182,...,2.074975,2.584545,2.074975,2.450491,2.697953,2.316318,2.421034,2.327056,2.305521,2.316318
2,1.088222,1.732480,1.674824,1.235654,0.836471,1.315006,1.484309,1.545443,2.006744,0.707240,...,1.811865,1.974349,1.811865,0.854473,1.301124,1.092894,1.155690,1.098620,1.087138,1.092894
3,1.880941,2.504997,2.535388,1.880900,1.544437,2.014078,2.423544,2.542821,2.784935,0.742334,...,0.706776,1.856629,0.706776,1.714065,2.358324,2.080343,1.900361,2.088160,2.072484,2.080343
4,1.659528,2.097781,2.593391,1.309826,1.316238,1.668676,2.019882,2.131893,2.546386,0.675713,...,1.153538,1.286182,1.153538,1.050297,2.077151,1.793208,1.525407,1.801015,1.785358,1.793208


In [11]:
check_dataset(df_join_raw_log)

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 24312
Count zero:	 1
Count positive:	 563639
Count greater than 1:	 478820
Count less than -1:	 3371


In [12]:
# split graph in groups and subgroups

""" def split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id, by_group=False):
	list_df_groups_subgroups = []
	for group in groups_id:
		df_aux = df_join_raw_log.filter(like=group)
		list_aux = []
		
		if by_group:
			list_aux.append(df_aux)
		else:
			for subgroup in subgroups_id[group]:
				list_aux.append(df_aux.filter(like="{}_{}.".format(group, subgroup)))
		list_df_groups_subgroups.append(list_aux)
	return list_df_groups_subgroups """

dict_df_groups_subgroups = split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id)
dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,AR_1.1,AR_1.2,AR_1.3
0,0.067434,0.524123,-0.866715
1,2.617854,3.259370,4.805380
2,1.088222,1.732480,1.674824
3,1.880941,2.504997,2.535388
4,1.659528,2.097781,2.593391


In [13]:
check_dataset(dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 686
Count zero:	 0
Count positive:	 15646
Count greater than 1:	 13389
Count less than -1:	 146


In [14]:
# No apply transpose for kneighbors_graph
# dict_groups_subgroups_t = dict_df_groups_subgroups.copy()

# Aplly Transpose
dict_groups_subgroups_t = transpose_global(dict_df_groups_subgroups)

dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,0,1,2,3,4,5,6,7,8,9,...,5434,5435,5436,5437,5438,5439,5440,5441,5442,5443
0,0.067434,2.617854,1.088222,1.880941,1.659528,1.767491,2.008215,2.092772,2.072683,2.185935,...,2.207700,1.753761,0.217247,1.533782,1.191297,0.892378,0.916754,1.151250,1.478938,0.344954
1,0.524123,3.259370,1.732480,2.504997,2.097781,2.185932,2.417997,2.472316,2.501865,2.492657,...,2.999874,2.300622,0.948713,2.215614,1.527121,3.759797,1.620635,2.041729,2.421294,0.992325
2,-0.866715,4.805380,1.674824,2.535388,2.593391,2.632587,2.750429,2.788159,2.777456,2.834100,...,3.664453,3.152539,2.012696,2.671736,2.787052,0.585675,0.718502,2.984408,3.121924,0.136437


In [15]:
check_dataset(dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 686
Count zero:	 0
Count positive:	 15646
Count greater than 1:	 13389
Count less than -1:	 146


In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import kneighbors_graph

def correlation_ledoitwolf_global(exp, dict_groups_subgroups_t):
	dict_groups_subgroups_t_corr = {}
	for group_id, dict_groups in dict_groups_subgroups_t.items():
		dict_aux = {}
		for subgroup_id, df_subgroup in dict_groups.items():
			print(group_id, subgroup_id, df_subgroup.shape)
			
			""" import numpy as np
			cov = np.cov(df_subgroup.values, rowvar=False)
			cond = np.linalg.cond(cov)
			print("Condition number:", cond, cond > 1e8) # ill-conditioned if > 1e8 (True, instable) """

			scaler = StandardScaler()
			df_subgroup_scaled = scaler.fit_transform(df_subgroup)
			lw = LedoitWolf()
			lw.fit(df_subgroup_scaled)

			# Matriz de covarianza regularizada
			cov = lw.covariance_
			std = np.sqrt(np.diag(cov))
			corr = cov / np.outer(std, std)
			matrix = pd.DataFrame(corr)
		
			dict_aux[subgroup_id] = matrix
			
			matrix.to_csv("experiments/output/{}/correlations/{}_{}.csv".format(exp, group_id, subgroup_id), index=True)
		dict_groups_subgroups_t_corr[group_id] = dict_aux
	return dict_groups_subgroups_t_corr

def correlation_kneighbors_graph_global(exp, dict_groups_subgroups_t):
	dict_groups_subgroups_t_corr = {}
	for group_id, dict_groups in dict_groups_subgroups_t.items():
		dict_aux = {}
		for subgroup_id, df_subgroup in dict_groups.items():
			print(group_id, subgroup_id, df_subgroup.shape)
			
			""" import numpy as np
			cov = np.cov(df_subgroup.values, rowvar=False)
			cond = np.linalg.cond(cov)
			print("Condition number:", cond, cond > 1e8) # ill-conditioned if > 1e8 (True, instable) """
			
			k = 10
			scaler = StandardScaler()
			df_subgroup_scaled = scaler.fit_transform(df_subgroup)
			A = kneighbors_graph(
				df_subgroup_scaled, # X, X_scaled
				n_neighbors=k,
				metric="euclidean", # "cosine",
				mode="distance",
				include_self=True
			)
			matrix = pd.DataFrame(A.toarray())
		
			dict_aux[subgroup_id] = matrix
			
			matrix.to_csv("experiments/output/{}/correlations/{}_{}.csv".format(exp, group_id, subgroup_id), index=True)
		dict_groups_subgroups_t_corr[group_id] = dict_aux
	return dict_groups_subgroups_t_corr

In [17]:
# Correlation matrix (partial correlation)

# Option 1
# dict_groups_subgroups_t_corr = correlation_global(exp, dict_groups_subgroups_t)

# Option 2
dict_groups_subgroups_t_corr = correlation_ledoitwolf_global(exp, dict_groups_subgroups_t)

# Option 3
# dict_groups_subgroups_t_corr = correlation_kneighbors_graph_global(exp, dict_groups_subgroups_t)

dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

AR 1 (3, 5444)
AR 2 (3, 5444)
BC 1 (3, 5444)
BC 2 (3, 5444)
BPH 1 (3, 5444)
BPH 2 (3, 5444)
CKD14 1 (3, 5444)
CKD14 2 (3, 5444)
CKD5 1 (3, 5444)
CKD5 2 (3, 5444)
CRS 1 (3, 5444)
CRS 2 (3, 5444)
ED 1 (3, 5444)
ED 2 (3, 5444)
HC 1 (3, 5444)
HC 2 (3, 5444)
LP 1 (3, 5444)
LP 2 (3, 5444)
LPRD 1 (3, 5444)
LPRD 2 (3, 5444)
LSNB 1 (3, 5444)
LSNB 2 (3, 5444)
OSA 1 (3, 5444)
OSA 2 (3, 5444)
PCa 1 (3, 5444)
PCa 2 (3, 5444)
PD 1 (3, 5444)
PD 2 (3, 5444)
PSG 1 (3, 5444)
PSG 2 (3, 5444)
RCC 1 (3, 5444)
RCC 2 (3, 5444)
SGB 1 (3, 5444)
SGB 2 (3, 5444)
health 1 (3, 5444)
health 2 (3, 5444)


,0,1,2,3,4,5,6,7,8,9,...,5434,5435,5436,5437,5438,5439,5440,5441,5442,5443
0,1.000000,-0.581910,-0.081671,-0.167403,-0.488842,-0.480137,-0.437025,-0.441106,-0.399423,-0.486483,...,-0.442403,-0.533492,-0.524522,-0.405987,-0.616070,0.575920,0.623235,-0.478874,-0.422993,0.631774
1,-0.581910,1.000000,0.476864,0.537968,0.699493,0.697054,0.682853,0.684338,0.667944,0.698847,...,0.684804,0.709319,0.707739,0.670700,0.710778,-0.225843,-0.307141,0.696687,0.677544,-0.323237
2,-0.081671,0.476864,1.000000,0.708247,0.572343,0.579346,0.610385,0.607687,0.633125,0.574267,...,0.606820,0.531818,0.540636,0.629418,0.428181,0.352602,0.273869,0.580339,0.619307,0.257216
3,-0.167403,0.537968,0.708247,1.000000,0.619985,0.625770,0.650857,0.648717,0.668498,0.621579,...,0.648028,0.585805,0.593334,0.665675,0.494519,0.274417,0.191574,0.626587,0.657867,0.174232
4,-0.488842,0.699493,0.572343,0.619985,1.000000,0.713461,0.710306,0.710781,0.704376,0.713553,...,0.710924,0.710684,0.711758,0.705604,0.684330,-0.087652,-0.173830,0.713431,0.708412,-0.191174


In [18]:
# Check correlation matrices

dict_groups_subgroups_t_corr

{'AR': {'1':           0         1         2         3         4         5         6     \
  0     1.000000 -0.581910 -0.081671 -0.167403 -0.488842 -0.480137 -0.437025   
  1    -0.581910  1.000000  0.476864  0.537968  0.699493  0.697054  0.682853   
  2    -0.081671  0.476864  1.000000  0.708247  0.572343  0.579346  0.610385   
  3    -0.167403  0.537968  0.708247  1.000000  0.619985  0.625770  0.650857   
  4    -0.488842  0.699493  0.572343  0.619985  1.000000  0.713461  0.710306   
  ...        ...       ...       ...       ...       ...       ...       ...   
  5439  0.575920 -0.225843  0.352602  0.274417 -0.087652 -0.075872 -0.019697   
  5440  0.623235 -0.307141  0.273869  0.191574 -0.173830 -0.162305 -0.107016   
  5441 -0.478874  0.696687  0.580339  0.626587  0.713431  0.713558  0.711471   
  5442 -0.422993  0.677544  0.619307  0.657867  0.708412  0.709736  0.713343   
  5443  0.631774 -0.323237  0.257216  0.174232 -0.191174 -0.179724 -0.124720   
  
            7         8   

In [19]:
check_dataset(dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][1]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 11196088
Count zero:	 1152
Count positive:	 18439896
Count greater than 1:	 0
Count less than -1:	 0


In [ ]:
def build_graph_weight_global_directed_new(exp, dict_groups_subgroups_t_corr, threshold=0.3):
	dict_groups_subgroups_t_corr_g = {}
	for group_id, dict_groups in dict_groups_subgroups_t_corr.items():
		dict_aux = {}
		for subgroup_id, df_subgroup in dict_groups.items():
			# Percentil Correlaciones conservadas 	Densidad
			# 50	    50 %	                    Muy densa
            # 75	    25 %	                    Densa
            # 90	    10 %	                    Moderadamente dispersa
            # 95	    5 %	                        Dispersa
            # 99	    1 %	                        Muy dispersa
			threshold = np.percentile(np.abs(df_subgroup), 90)
			
			df_weighted_edges = (df_subgroup.where(np.triu(np.ones(df_subgroup.shape), k=1).astype(bool)).stack())
			df_weighted_edges = df_weighted_edges.dropna().to_frame()
			df_weighted_edges.reset_index(inplace=True)
			df_weighted_edges.columns = ["source", "target", "weight"]
			df_weighted_edges = df_weighted_edges[df_weighted_edges["weight"].abs() >= threshold]
			df_weighted_edges["subgroup"] = [subgroup_id] * len(df_weighted_edges)
			dict_aux[subgroup_id] = df_weighted_edges
			
			df_weighted_edges.to_csv("experiments/output/{}/preprocessing/edges/{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# G = nx.from_pandas_edgelist(df_weighted_edges, "source", "target", edge_attr=["weight"])
			# print(groups_id[i], subgroups_id[groups_id[i]][j], G.number_of_nodes(), G.number_of_edges())
			# nx.write_gexf(G, "experiments/output/{}/preprocessing/graphs/graphs_{}_{}.gexf".format(exp, groups_id[i], subgroups_id[groups_id[i]][j]))
		dict_groups_subgroups_t_corr_g[group_id] = dict_aux
	return dict_groups_subgroups_t_corr_g

In [21]:
# Build graph (corpus graphs)

# dict_groups_subgroups_t_corr_g = build_graph_weight_global_directed(exp, dict_groups_subgroups_t_corr, threshold=threshold_corr)
dict_groups_subgroups_t_corr_g = build_graph_weight_global_directed_new(exp, dict_groups_subgroups_t_corr, threshold=threshold_corr)
dict_groups_subgroups_t_corr_g[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,source,target,weight,subgroup
15,0,16,0.710738,1
35,0,36,0.710639,1
44,0,45,0.711307,1
48,0,49,0.713266,1
74,0,75,0.709949,1


In [22]:
def create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups, df_join_raw_metadata):	
	for group_id in tqdm(groups_id):
		for subgroup_id in tqdm(subgroups_id[group_id]):
			df_weighted_edges = pd.read_csv("experiments/output/{}/preprocessing/edges/{}_{}.csv".format(exp, group_id, subgroup_id))
			# print(df_weighted_edges)
			G = nx.from_pandas_edgelist(df_weighted_edges, "source", "target", edge_attr=["weight", "subgroup"])
			dict_id_idx = dict(zip(list(G.nodes()), range(G.number_of_nodes())))
			G = nx.relabel_nodes(G, dict_id_idx)

			df_nodes = dict_df_groups_subgroups[group_id][subgroup_id].loc[list(dict_id_idx.keys())] # A_1.1, A_1.2, A_1.3
			# from IPython.display import display
			# display(df_nodes)

			# nodes, with node features
			metadata = df_join_raw_metadata.loc[df_nodes.index] # Average Rt, Average Mz
			# intensity = df_join_raw_log.loc[df_nodes.index] # A_1.1, A_1.2, A_1.3, A_2.1, ...

			""" e = 1e-8
			mz = metadata.iloc[:, 1].values
			rt = metadata.iloc[:, 0].values
			intensity_mean = df_nodes.mean(axis=1).values
			intensity_std = df_nodes.std(axis=1).values
			intensity_cv = intensity_std / intensity_mean
			presence_ratio = (df_nodes > 0).mean(axis=1)

			mz_log = np.log10(mz + e)
			# intensity_mean_log = np.log10(intensity_mean + e)
			
			# mz_z = (mz_log - mz_log.mean()) / mz_log.std() # z-score
			rt_z = (rt - rt.mean()) / rt.std() # z-score
			# intensity_mean_z = (intensity_mean_log - intensity_mean_log.mean()) / intensity_mean_log.std() # z-score

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": mz_log,
				"rt": rt_z,
				"intensity_mean": intensity_mean,
				"intensity_std": intensity_std,
				"intensity_cv": intensity_cv,
				"presence_ratio": presence_ratio
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			# df_node_features.insert(0, "idx", list(dict_id_idx.values()))
			# df_node_features.insert(1, "id", list(dict_id_idx.keys()))
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features) """

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": metadata.iloc[:, 1].values,
				"rt": metadata.iloc[:, 0].values,
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features)

			# edges
			edges = list(G.edges())
			df_edges = pd.DataFrame(edges, columns=["source", "target"])
			df_edges["weight"] = [G.get_edge_data(*edge)["weight"] for edge in edges]
			df_edges["subgroup"] = [G.get_edge_data(*edge)["subgroup"] for edge in edges]
			df_edges.to_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)


In [23]:
# create dataset - nodes/edge data for PyTorch Geometric/DGL framework

# IMPORTANT
dict_df_groups_subgroups_ = split_groups_subgroups(df_join_raw_intensity, groups_id, subgroups_id) # Important (intesities without Log)

for data_variation in data_variations:
	if data_variation == "none":
		create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, df_join_raw_metadata)
	else:
		# dynamic graph to static graph
		create_graph_data_directed_variation(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, data_variation)

100%|██████████| 18/18 [04:17<00:00, 14.30s/it]


In [24]:
# details
list_details = []
	
for group_id in groups_id:
	subgroups_id_ = []
	for data_variation in data_variations:
		if data_variation == "none":
			subgroups_id_ += subgroups_id[group_id]
		else:
			subgroups_id_ += [data_variation]
	# print(subgroups)
	
	for subgroup_id_ in subgroups_id_:
		try:
			df_edges = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id_))

			G = nx.from_pandas_edgelist(df_edges.iloc[:, [0, 1]])
			list_details.append([group_id, subgroup_id_, G.number_of_nodes(), G.number_of_edges(), nx.density(G), np.nan, nx.is_connected(G)])
		except:
			list_details.append([group_id, subgroup_id_, G.number_of_nodes(), G.number_of_edges(), np.nan, np.nan, np.nan])

df_details = pd.DataFrame(list_details, columns=["Group", "Subgroup", "Num. nodes", "Num. edges", "Density", "Diameter", "Is connected"])
df_details.to_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp), index=False)

df_details = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp))
df_details

,Group,Subgroup,Num. nodes,Num. edges,Density,Diameter,Is connected
0,AR,1,5444,1479135,0.099835,NaN,True
1,AR,2,5444,1479135,0.099835,NaN,True
2,BC,1,5444,1479135,0.099835,NaN,True
3,BC,2,5444,1479135,0.099835,NaN,True
4,BPH,1,5444,1479135,0.099835,NaN,True
5,BPH,2,5444,1479135,0.099835,NaN,True
6,CKD14,1,5444,1479135,0.099835,NaN,True
7,CKD14,2,5444,1479135,0.099835,NaN,True
8,CKD5,1,5444,1479135,0.099835,NaN,True
9,CKD5,2,5444,1479135,0.099835,NaN,True
